# Import

In [ ]:
import os, zipfile, tempfile, random, keras.ops, cv2, joblib, keras, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import seaborn as sns
import tensorflow as tf
from utils import *
from PIL import Image
from pathlib import Path

from tensorflow.keras import layers, models, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import backend as K
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tfswin import SwinTransformerTiny224

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score, recall_score, accuracy_score, precision_score, roc_auc_score, silhouette_score, pairwise_distances, normalized_mutual_info_score, adjusted_rand_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.cluster import KMeans
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import LabelEncoder
keras.config.enable_unsafe_deserialization()

# Suppress TF logs
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async' 
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ['PYTHONHASHSEED'] = '42'
tf.get_logger().setLevel('ERROR')
random.seed(42)
np.random.seed(42)
tf.keras.utils.set_random_seed(42) 

# GPU config
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
print(f" TensorFlow {tf.__version__} | GPUs: {len(gpus)}")

# Internal Dataset

## Initiation

In [ ]:
# Hyperparameter
MODEL_NAME = "your_model_name"  # Replace with your model name
VERSION    = "v1"  # Replace with your version
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
EPOCHS     = 250
LR         = 5e-5
PATIENCE   = 20

# Paths
DATA_ROOT  = "your_data_root_path"  # Replace with your data root path
TRAIN_DIR  = os.path.join(DATA_ROOT, "train")
VAL_DIR    = os.path.join(DATA_ROOT, "val")
TEST_DIR   = os.path.join(DATA_ROOT, "test")

CKPT_DIR   = "your_checkpoint_dir"  # Replace with your checkpoint directory
BASE_RESULT_DIR = "your_base_result_dir"  # Replace with your base result directory
RESULT_DIR      = os.path.join(BASE_RESULT_DIR, MODEL_NAME)
os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"Results will be saved to → {RESULT_DIR}")

MODEL_PATH = os.path.join(CKPT_DIR, f"{MODEL_NAME}.keras")
CSV_LOG    = os.path.join(BASE_RESULT_DIR, "experiment_results.csv")

In [ ]:
# Focal loss
class_weights_alpha = [0.15, 0.35, 0.5]  # benign, malignant, normal
focal_loss = tf.keras.losses.CategoricalFocalCrossentropy(
    gamma=2.0, 
    alpha=class_weights_alpha
)
print("Focal Loss initialized")

## Model DL Setup

### Shallow CNN

In [ ]:
# Train: Augmentation + Rescale
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20, width_shift_range=0.2, height_shift_range=0.2,
    shear_range=0.2, zoom_range=0.2, horizontal_flip=True, fill_mode='nearest'
)

# Val/Test: Rescale ONLY
val_datagen  = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=True
)
val_data = val_datagen.flow_from_directory(
    VAL_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)
test_data = test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

CLASS_NAMES = list(train_data.class_indices.keys())
NUM_CLASSES = len(CLASS_NAMES)
print(f"Classes: {CLASS_NAMES}")
print(f"Samples → Train: {train_data.samples} | Val: {val_data.samples} | Test: {test_data.samples}")

In [ ]:
def build_shallow_cnn():
    return models.Sequential([
        Input(shape=(224, 224, 3)),
        # Layer 1: Conv + Pool
        layers.Conv2D(32, (5, 5), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        # Layer 2: Conv + Pool
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        # Layer 3: Conv + Pool
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        # Flatten + Dense + Dropout + Output
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ], name="ShallowCNN")

model = build_shallow_cnn()
model.summary()

### Resnet 50

In [ ]:
# Preprocessing for ResNet50
test_datagen_rn = ImageDataGenerator(preprocessing_function=preprocess_input)
train_datagen_rn = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20, width_shift_range=0.2, height_shift_range=0.2,
    shear_range=0.2, zoom_range=0.2, horizontal_flip=True, fill_mode='nearest'
)
val_datagen_rn = ImageDataGenerator(preprocessing_function=preprocess_input)

train_data_rn = train_datagen_rn.flow_from_directory(TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=True)
val_data_rn   = val_datagen_rn.flow_from_directory(VAL_DIR,   target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)
test_data_rn  = test_datagen_rn.flow_from_directory(TEST_DIR,  target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)

# Hyperparameters for ResNet50
UNFREEZE_LAYERS = 15    # 0=frozen extractor | 10=fine-tune top 10 | -1=full
HEAD_UNITS      = 128  # 0 = GAP → Dropout → softmax directly
CLASS_NAMES = list(train_data_rn.class_indices.keys())
NUM_CLASSES = len(CLASS_NAMES)
print(f"Classes: {CLASS_NAMES}")
print(f"Samples → Train: {train_data_rn.samples} | Val: {val_data_rn.samples} | Test: {test_data_rn.samples}")

In [ ]:
def build_resnet50():
    base = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base.trainable = False
    # Feature extractor: Unfreeze last N layers (except BatchNorm)
    if UNFREEZE_LAYERS == -1:
      base.trainable = True
      for layer in base.layers:
          if isinstance(layer, layers.BatchNormalization):
              layer.trainable = False   # keep BN frozen
    elif UNFREEZE_LAYERS > 0:
        for layer in base.layers[-UNFREEZE_LAYERS:]:
            if not isinstance(layer, layers.BatchNormalization):
                layer.trainable = True
    for i, layer in enumerate(base.layers):
        print(f"[{i}] {layer.name:40s} trainable={layer.trainable}")
        if 'conv4_block1_1_conv' in layer.name or \
        'conv5_block3_out' in layer.name or \
        'conv4_block1_1_bn' in layer.name:
            print(f"{layer.name}: trainable={layer.trainable}")
    
    # Build the classification head
    inputs = Input(shape=(224, 224, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dense(256, activation='relu', name='dense_1')(x)
    x = layers.BatchNormalization(name='bn_head')(x)
    x = layers.Dropout(0.3, name='dropout_1')(x)   
    x = layers.Dense(128, activation='relu', name='dense_2')(x)
    x = layers.Dropout(0.2, name='dropout_2')(x)  
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', name='predictions')(x)

    return models.Model(inputs, outputs, name='ResNet50Extractor')


model = build_resnet50()
model.summary()

### Swin Tiny

In [ ]:
# Swin preprocessing is handled inside the model using Lambda layer
# So generators just need 1/255 rescaling
train_datagen_vit = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20, width_shift_range=0.2, height_shift_range=0.2,
    shear_range=0.2, zoom_range=0.2, horizontal_flip=True, fill_mode='nearest'
)
val_datagen_vit  = ImageDataGenerator(rescale=1./255)
test_datagen_vit = ImageDataGenerator(rescale=1./255)

train_data_vit = train_datagen_vit.flow_from_directory(TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=True)
val_data_vit   = val_datagen_vit.flow_from_directory(VAL_DIR,   target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)
test_data_vit  = test_datagen_vit.flow_from_directory(TEST_DIR,  target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)

UNFREEZE_TRANSFORMER_BLOCKS = 1    # unfreeze last N transformer encoder blocks
CLASS_NAMES = list(train_data_vit.class_indices.keys())
NUM_CLASSES = len(CLASS_NAMES)
print(f"Classes: {CLASS_NAMES}")
print(f"Samples → Train: {train_data_vit.samples} | Val: {val_data_vit.samples} | Test: {test_data_vit.samples}")

In [ ]:
def build_swin_tiny():
    # Feature extractor: SwinTransformerTiny224
    base = SwinTransformerTiny224(include_top=False)
    base.trainable = False
    for layer in base.layers[-UNFREEZE_TRANSFORMER_BLOCKS:]:
        if not isinstance(layer, layers.LayerNormalization):
            layer.trainable = True
    for i, layer in enumerate(base.layers):
        print(f"[{i}] {layer.name:40s} trainable={layer.trainable}")
    # Build the classification head
    inputs = Input(shape=(224, 224, 3), name='pixel_input')
    x = layers.Rescaling(255.0, name='to_uint8')(inputs)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
    x = layers.Dense(256, activation='relu', name='dense_1')(x)
    x = layers.BatchNormalization(name='bn_head')(x)
    x = layers.Dropout(0.3, name='dropout_1')(x)
    x = layers.Dense(128, activation='relu', name='dense_2')(x)
    x = layers.Dropout(0.2, name='dropout_2')(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', name='predictions')(x)

    return models.Model(inputs=inputs, outputs=outputs, name='SwinTiny')

model = build_swin_tiny()
model.summary()

## Train DL

In [ ]:
# Setup optimizer, loss, and metrics
optimizer = tf.keras.optimizers.Adam(learning_rate=LR)

model.compile(
    optimizer=optimizer,
    loss=focal_loss,
    # loss= tf.keras.losses.CategoricalCrossentropy(),
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True, verbose=1),
    ModelCheckpoint(filepath=MODEL_PATH, monitor='val_loss', save_best_only=True, verbose=1),
    # ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=12, min_lr=1e-7, verbose=1)
]
print("Model compiled & callbacks ready")

In [ ]:
print(f"Training {MODEL_NAME}...")
history = model.fit( 
    train_data_rn, # Use training data from the generator
    epochs=EPOCHS,
    validation_data=val_data_rn, # Use validation data from the generator
    callbacks=callbacks,
    verbose=1,
)
print(f"Done. Best val_loss: {min(history.history['val_loss']):.4f} | Epochs: {len(history.history['loss'])}")

In [ ]:
# Plot training curves
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('Accuracy'); plt.legend(); plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss'); plt.legend(); plt.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, f"{MODEL_NAME}_curves.png"), dpi=150)
plt.show()

## Load Model DL

In [ ]:
# Load Model for model non swin
model = load_model(
    os.path.join(CKPT_DIR, "resnet50_focal_v4.keras"),
    custom_objects={"CategoricalFocalCrossentropy": tf.keras.losses.CategoricalFocalCrossentropy},
    
)
print(f" Model loaded from → {MODEL_PATH}")
model.summary()

In [ ]:
# Load Swin-Tiny model and weights
model = build_swin_tiny()   # rebuild the architecture

model.load_weights(MODEL_PATH)  # load weights only, skip config deserialization
print(f"Swin-Tiny weights loaded from → {MODEL_PATH}")

## Evaluation

In [ ]:
y_true = test_data_rn.classes # Use the true labels from the test data generator
predictions = model.predict(test_data_rn) # use the test data generator for predictions
y_pred = np.argmax(predictions, axis=1)

print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, labels=[0, 1, 2], target_names=CLASS_NAMES, digits=4))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix'); plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, f"{MODEL_NAME}_cm.png"), dpi=150)
plt.show()

# Macro AUC
y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
macro_auc = roc_auc_score(y_true_bin, predictions, average='macro', multi_class='ovr')
print(f"✅ Macro OvR AUC: {macro_auc:.4f}")

In [ ]:
# Save results to CSV
row = {
    "model": MODEL_NAME,
    "val_acc": max(history.history['val_accuracy']),
    "val_loss": min(history.history['val_loss']),
    "test_acc": np.mean(y_true == y_pred),
    "macro_auc": macro_auc,
    "epochs": len(history.history['loss']),
    "lr": LR,
    "patience": PATIENCE
}
df = pd.DataFrame([row])
if os.path.exists(CSV_LOG):
    df.to_csv(CSV_LOG, mode='a', header=False, index=False)
else:
    df.to_csv(CSV_LOG, index=False)

print(f"Results saved to {CSV_LOG}")
print(pd.read_csv(CSV_LOG).to_string(index=False))

## Gradcam ++

In [ ]:
# Collect Misclassified Samples
# Runs a full forward pass over test_data, returns up to max_per_class wrong
# predictions per true class (for balanced coverage across all classes)

misclassified = collect_misclassified(
    model,
    test_data_rn, # Use the test data generator for predictions
    class_names=CLASS_NAMES,
    max_per_class=4,          # adjust: more = larger figure
)

total_wrong = len(misclassified)
total_test  = test_data_rn.samples # Use the total number of samples in the test data generator
print(f"❌ Misclassified: {total_wrong} / {total_test}  "
      f"({total_wrong/total_test:.1%} error rate)")

# Breakdown by true class
from collections import Counter
true_counts = Counter(s['true_label'] for s in misclassified)
for cls, cnt in true_counts.items():
    print(f"   {cls}: {cnt} misclassified samples collected")

In [ ]:
# Misclassification Pair Summary
plot_misclassification_summary(
    misclassified,
    CLASS_NAMES,
    save_path=os.path.join(RESULT_DIR, f"{MODEL_NAME}_misclass_pairs.png"),
)

In [ ]:
import importlib
import utils
importlib.reload(utils)  # wipes all previous patches, restores true original

_true_gradcam = utils.get_gradcam_pp  # capture in closure — immune to future patches

def smart_gradcam(model, img, cls, layer=None):
    is_swin = any(
        'swin' in l.name.lower() or 'tiny' in l.name.lower()
        for l in model.layers
    )
    if is_swin:
        return utils.get_gradcam_pp_swin(model, img, cls)
    return _true_gradcam(model, img, cls, layer)  # calls real fn, not utils.get_gradcam_pp

utils.get_gradcam_pp = smart_gradcam

plot_misclassified_gradcam(
    model,
    misclassified,
    last_conv_layer_name=None,
    max_display=12,
    save_path=os.path.join(RESULT_DIR, f"{MODEL_NAME}_gradcam_misclassified.png"),
)

utils.get_gradcam_pp = _true_gradcam  # restore

## Train Model ML

In [ ]:
BACKBONE    = "swin" # "swin" (768-d, in-scope) or "resnet" (2048-d)

In [ ]:
#  Build feature extractor + matching preprocessing 
if BACKBONE == "swin": # swin: 768-d features, in-scope
    swin = build_swin_tiny()
    _ = swin(np.zeros((1, 224, 224, 3), dtype=np.float32), training=False)
    with zipfile.ZipFile(os.path.join(CKPT_DIR, "your_swin_checkpoint.keras"), 'r') as zf:
        with tempfile.TemporaryDirectory() as tmp:
            zf.extractall(tmp)
            swin.load_weights(os.path.join(tmp, 'model.weights.h5'))
    feature_extractor = tf.keras.Model(swin.input,
                                       swin.get_layer('global_avg_pool').output)
    _preprocess = dict(rescale=1./255) # Swin: [0,1] generator
    
else:  # resnet: 2048-d features, out-of-scope
    rn = tf.keras.models.load_model(
        os.path.join(CKPT_DIR, "your_resnet_checkpoint.keras"),
        custom_objects={"CategoricalFocalCrossentropy":
                        tf.keras.losses.CategoricalFocalCrossentropy})
    feature_extractor = tf.keras.Model(rn.input, rn.get_layer('gap').output)
    _preprocess = dict(preprocessing_function=resnet_preprocess)
print(f"Feature extractor ({BACKBONE}) — output dim: {feature_extractor.output_shape}")

def extract_features(data_dir):
    flow = ImageDataGenerator(**_preprocess).flow_from_directory(
        data_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode='categorical', shuffle=False)
    return feature_extractor.predict(flow, verbose=1), flow.classes

# get features for train, val, test
print("Extracting features ...")
X_train, y_train = extract_features(TRAIN_DIR)
X_val,   y_val   = extract_features(VAL_DIR)
X_test,  y_test  = extract_features(TEST_DIR)

In [ ]:
# Helper: wrap an estimator in a scaler-pipeline only if it needs scaling 
def make_model(estimator, needs_scaling):
    """Scale-sensitive : Pipeline([scaler, est]); scale-invariant : est as-is.
    The StandardScaler is fit on train+val inside .fit(), so the saved model
    carries its own preprocessing and test-time scaling is automatic."""
    if needs_scaling:
        return Pipeline([("scaler", StandardScaler()), ("clf", estimator)])
    return estimator

# (estimator, needs_scaling)
SPECS = {
    "SVM":                (SVC(kernel='rbf', C=10, gamma='scale', probability=True,
                              random_state=42), True),
    "Random_Forest":      (RandomForestClassifier(n_estimators=200, random_state=42,
                              n_jobs=-1), False),
    "XGBoost":            (XGBClassifier(n_estimators=200, learning_rate=0.05,
                              random_state=42, eval_metric='mlogloss',
                              verbosity=0), False),
    "KNN":                (KNeighborsClassifier(n_neighbors=7, metric='cosine'), True),
    "Naive_Bayes":        (GaussianNB(), False),
    "Decision_Tree":      (DecisionTreeClassifier(max_depth=10, random_state=42), False),
    "Logistic_Regression":(LogisticRegression(max_iter=1000, C=1.0, random_state=42), True),
}

results = []
for name, (est, needs_scaling) in SPECS.items():
    print(f"\n🔄 Training {name}  (scaled={needs_scaling}) ...")
    clf = make_model(est, needs_scaling)
    clf.fit(X_train, y_train)                    
    y_pred  = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)
    acc = float(np.mean(y_pred == y_test))
    auc = roc_auc_score(label_binarize(y_test, classes=list(range(NUM_CLASSES))),
                        y_proba, average='macro', multi_class='ovr')
    print(classification_report(y_test, y_pred, labels=[0, 1, 2], target_names=CLASS_NAMES, digits=4))

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f'{name} — Confusion Matrix')
    plt.ylabel('True'); plt.xlabel('Predicted'); plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, f"{name}_cm.png"), dpi=150); plt.show()

    joblib.dump(clf, os.path.join(CKPT_DIR, f"{name}_{VERSION}.joblib"))
    results.append({"model": name, "scaled": needs_scaling,
                    "test_acc": round(acc, 4), "macro_auc": round(auc, 4)})

In [ ]:
# Leaderboard
df_results = pd.DataFrame(results).sort_values("macro_auc", ascending=False)
print("\n📊 Classical ML Summary:")
print(df_results.to_string(index=False))
df_results.to_csv(os.path.join(RESULT_DIR, "classical_ml_results.csv"), index=False)
print(f"\n Models → {CKPT_DIR}/*_{VERSION}.joblib")
print(f"Results → {RESULT_DIR}/classical_ml_results.csv")

## Ensemble

In [ ]:
# Helper
def load_dl_model(path, custom_objects=None):
    return tf.keras.models.load_model(
        path,
        custom_objects=custom_objects or {
            "CategoricalCrossentropy": tf.keras.losses.CategoricalCrossentropy,
        },
    )

def load_swin_model(path):
    model = build_swin_tiny()
    dummy = np.zeros((1, 224, 224, 3), dtype=np.float32)
    _ = model(dummy, training=False)
    with zipfile.ZipFile(path, 'r') as zf:
        with tempfile.TemporaryDirectory() as tmpdir:
            zf.extractall(tmpdir)
            model.load_weights(os.path.join(tmpdir, 'model.weights.h5'))
    return model


def get_dl_preds(model, generator):
    generator.reset()
    return model.predict(generator, verbose=1)

def get_classical_preds(clf_path, features):
    clf = joblib.load(clf_path)
    return clf.predict_proba(features)


def _clf_preds(path, X):
    return joblib.load(path).predict_proba(X)

def _metrics(proba, y_true):
    y_pred = np.argmax(proba, axis=1)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    acc  = np.mean(y_pred == y_true)
    wf1  = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    mr   = cm[1, 1] / cm[1].sum() if cm[1].sum() else 0.0
    br   = cm[0, 0] / cm[0].sum() if cm[0].sum() else 0.0
    nr   = cm[2, 2] / cm[2].sum() if cm[2].sum() else 0.0
    return acc, wf1, mr, br, nr, cm

def get_malignant_recall(y_true, y_proba):
    y_pred = np.argmax(y_proba, axis=1)
    recalls = recall_score(y_true, y_pred, average=None, zero_division=0)
    return recalls[1] 

def pad_probs(p):
    if p.shape[1] == 2:
        return np.pad(p, ((0, 0), (0, 1)), mode='constant', constant_values=0.0)
    return p


In [ ]:
# Load candidate pool — test & val predictions
# Prerequisites:
#   • build_swin_tiny()
#   • val_data_vit, test_data_vit, val_data_rn, test_data_rn 
#   • X_val, X_test (Swin features)

#  Check X_val in scope 
try:
    print(f"✅ X_val shape: {X_val.shape} | X_test shape: {X_test.shape}")
except NameError:
    raise RuntimeError("❌ X_val / X_test not found. Run extraction using Swin extractor first.")

# DL models — load → predict → delete to free GPU memory
print("\n" + "="*60 + "\n  DL MODELS\n" + "="*60)

print("\n[1/3] swin_tiny_1")
_m = load_swin_model(os.path.join(CKPT_DIR, "your_swin_checkpoint.keras"))
TEST_swin_v1 = get_dl_preds(_m, test_data_vit);  VAL_swin_v1 = get_dl_preds(_m, val_data_vit)
del _m

print("\n[2/3] swin_tiny_2")
_m = load_swin_model(os.path.join(CKPT_DIR, "your_swin_checkpoint.keras"))
TEST_swin_v2 = get_dl_preds(_m, test_data_vit);  VAL_swin_v2 = get_dl_preds(_m, val_data_vit)
del _m

print("\n[3/3] resnet50_focal_v4")
_m = load_dl_model(
    os.path.join(CKPT_DIR, "your_resnet_checkpoint.keras"),
    custom_objects={"CategoricalFocalCrossentropy": tf.keras.losses.CategoricalFocalCrossentropy})
TEST_rn_v4 = get_dl_preds(_m, test_data_rn);  VAL_rn_v4 = get_dl_preds(_m, val_data_rn)
del _m

In [ ]:
# ML  models (Swin features — X_val already in scope)
print("\n" + "="*60 + "\n  ML MODELS\n" + "="*60)

_C = {
    "LR":      "your_lr_checkpoint.joblib",
    "RF":      "your_rf_checkpoint.joblib",
    "XGB":     "your_xgb_checkpoint.joblib",
    "SVM":     "your_svm_checkpoint.joblib",
    "KNN":     "your_knn_checkpoint.joblib",
}
_test_clfs, _val_clfs = {}, {}
for tag, fname in _C.items():
    path = os.path.join(CKPT_DIR, fname)
    _test_clfs[tag] = _clf_preds(path, X_test)
    _val_clfs[tag]  = _clf_preds(path, X_val)
    print(f"  ✅ {fname}")

TEST_lr      = _test_clfs["LR"];       VAL_lr      = _val_clfs["LR"]
TEST_rf      = _test_clfs["RF"];       VAL_rf      = _val_clfs["RF"]
TEST_xgb    = _test_clfs["XGB"];      VAL_xgb     = _val_clfs["XGB"]
TEST_svm     = _test_clfs["SVM"];      VAL_svm     = _val_clfs["SVM"]
TEST_knn     = _test_clfs["KNN"];      VAL_knn     = _val_clfs["KNN"]

#True labels
y_test_true = test_data_vit.classes
y_val_true  = val_data_vit.classes

print(f"\nAll 10 models loaded — test: {len(y_test_true)} | val: {len(y_val_true)}")

In [ ]:
# Exhaustive combination search 
# Methods: soft_equal | soft_malw (malignant-recall weighted) | hard_vote
# Output:  leaderboard sorted by validation malignant_recall → validation accuracy

# Pool of candidate models (test & val predictions + validation malignant recall)
POOL = {
    "swin_tiny_v1":      (pad_probs(TEST_swin_v1), get_malignant_recall(y_val_true, pad_probs(VAL_swin_v1))),
    "swin_tiny_v2":      (pad_probs(TEST_swin_v2), get_malignant_recall(y_val_true, pad_probs(VAL_swin_v2))),
    "resnet50_focal_v4": (pad_probs(TEST_rn_v4),   get_malignant_recall(y_val_true, pad_probs(VAL_rn_v4))),
    "LR":             (pad_probs(TEST_lr),   get_malignant_recall(y_val_true, pad_probs(VAL_lr))),
    "RF":             (pad_probs(TEST_rf),   get_malignant_recall(y_val_true, pad_probs(VAL_rf))),
    "XGBoost":        (pad_probs(TEST_xgb),  get_malignant_recall(y_val_true, pad_probs(VAL_xgb))),
    "SVM":            (pad_probs(TEST_svm),  get_malignant_recall(y_val_true, pad_probs(VAL_svm))),
    "KNN":            (pad_probs(TEST_knn),  get_malignant_recall(y_val_true, pad_probs(VAL_knn))),
}

VAL_POOL = {
    "swin_tiny_v1":      pad_probs(VAL_swin_v1),
    "swin_tiny_v2":      pad_probs(VAL_swin_v2),
    "resnet50_focal_v4": pad_probs(VAL_rn_v4),
    "LR":             pad_probs(VAL_lr),
    "RF":             pad_probs(VAL_rf),
    "XGBoost":        pad_probs(VAL_xgb),
    "SVM":            pad_probs(VAL_svm),
    "KNN":            pad_probs(VAL_knn),
}

_names      = list(POOL.keys())
_test_preds = [POOL[n][0] for n in _names]         # Original test predictions
_malw       = [POOL[n][1] for n in _names]         # Validation recalls (weights)
_val_preds  = [VAL_POOL[n] for n in _names]        # New validation predictions

# Voting functions
def _soft_equal(ps):
    return np.mean(np.stack(ps, 0), 0)

def _hard_vote(ps):
    hard = np.stack([np.argmax(p, 1) for p in ps], 1)     # (N, M)
    voted = np.array([np.bincount(row, minlength=3).argmax() for row in hard])
    out = np.zeros((len(voted), 3)); out[np.arange(len(voted)), voted] = 1.0
    return out

# Run search
rows = []
total = sum(len(list(itertools.combinations(_names, k))) for k in range(2, 7))
print(f"🔍 Evaluating {total} combos × 2 methods = {total*2} total...\n")

for k in range(2, 7):
    for combo in itertools.combinations(_names, k):
        idx   = [_names.index(n) for n in combo]
        ps_v  = [_val_preds[i] for i in idx]
        ps_t  = [_test_preds[i] for i in idx]
        ws    = [_malw[i]  for i in idx]
        tag   = "+".join(combo)

        for mname in ["soft_equal", "hard_vote"]:
            if mname == "soft_equal":
                proba_v = _soft_equal(ps_v)
                proba_t = _soft_equal(ps_t)
            elif mname == "hard_vote":
                proba_v = _hard_vote(ps_v)
                proba_t = _hard_vote(ps_t)

            # Evaluate on Validation Data
            v_acc, v_wf1, v_mr, v_br, v_nr, _ = _metrics(proba_v, y_val_true)
            # Evaluate on Test Data
            t_acc, t_wf1, t_mr, t_br, t_nr, _ = _metrics(proba_t, y_test_true)

            rows.append({
                "method": mname, "k": k, "combo": tag,
                # Validation Metrics
                "val_malignant_recall":  round(v_mr, 4),
                "val_accuracy":          round(v_acc, 4),
                "val_weighted_f1":       round(v_wf1, 4),
                "val_benign_recall":     round(v_br, 4),
                "val_normal_recall":     round(v_nr, 4),
                # Test Metrics (for reference)
                "test_malignant_recall": round(t_mr, 4),
                "test_accuracy":         round(t_acc, 4),
                "test_weighted_f1":      round(t_wf1, 4),
                "test_benign_recall":    round(t_br, 4),
                "test_normal_recall":    round(t_nr, 4),
            })

# Sort by validation performance
df_search = (pd.DataFrame(rows)
                .sort_values(["val_malignant_recall", "val_accuracy"], ascending=[False, False])
                .reset_index(drop=True))

# Leaderboard
DISPLAY_COLS = [
    "method", "k",
    "val_malignant_recall", "val_accuracy", "val_weighted_f1",
    "test_malignant_recall", "test_accuracy", "combo"
]

print("📊 TOP 25 — sorted by validation metrics:\n")
print(df_search[DISPLAY_COLS][df_search["k"] >= 4].head(25).to_string(index=True))

print("\n🏆 Best per method (by validation performance):")
for m in ["soft_equal",  "hard_vote"]:
    b = df_search[df_search["method"] == m].iloc[0]
    print(f"\n  [{m}]  [Val] malignant={b.val_malignant_recall:.4f} | acc={b.val_accuracy:.4f} | k={b.k}")
    print(f"         [Test] malignant={b.test_malignant_recall:.4f} | acc={b.test_accuracy:.4f}")
    print(f"         {b.combo}")

# Save
out_path = os.path.join(RESULT_DIR, "ensemble_exhaustive_search.csv")
df_search.to_csv(out_path, index=False)
print(f"\n💾 Saved → {out_path}")

In [ ]:
top_soft_vote_names = [
# use the best soft vote combination from the leaderboard
]
# Fetch padded probabilities directly from the POOL
top_soft_vote_probs = [POOL[name][0] for name in top_soft_vote_names]

# Equal soft voting calculation (simple average across all models)
stacked_probs = np.stack(top_soft_vote_probs, axis=0)
softvote_proba = np.mean(stacked_probs, axis=0) 

y_pred_soft_vote = np.argmax(softvote_proba, axis=1)

# Evaluation
cm_soft_vote = confusion_matrix(
    y_test_true,
    y_pred_soft_vote,
    labels=[0, 1, 2]
)

print("=" * 60)
print("SOFT VOTE ENSEMBLE — Top Combination")
print(f"Models: {' + '.join(top_soft_vote_names)}")
print("=" * 60)
print(cm_soft_vote)

print("\nClassification Report:")
print(classification_report(
    y_test_true,
    y_pred_soft_vote,
    labels=[0, 1, 2],
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
))

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm_soft_vote,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES
)
plt.title("Soft Vote Ensemble — Top Combination")
plt.ylabel("True")
plt.xlabel("Predicted")
plt.tight_layout()
save_path = os.path.join(RESULT_DIR, "soft_vote_top_combo_cm_internal.png")
plt.savefig(save_path, dpi=150)
print(f"\n💾 Confusion matrix saved → {save_path}")
plt.show()

In [ ]:
# Confusion Matrix: Top Hard-Vote Ensemble 
# Top hard-vote combo from exhaustive search:

top_hard_vote_models = [
# use the best hard vote combination from the leaderboard
]

hard_preds = np.stack(
    [np.argmax(p, axis=1) for p in top_hard_vote_models],
    axis=1
)

y_pred_hard_vote = np.array([
    np.bincount(row, minlength=3).argmax()
    for row in hard_preds
])

cm_hard_vote = confusion_matrix(
    y_test_true,
    y_pred_hard_vote,
    labels=[0, 1, 2]
)

print("=" * 60)
print("HARD VOTE ENSEMBLE — Top Combination")
print(f"Models: {' + '.join(top_hard_vote_models)}")
print("=" * 60)
print(cm_hard_vote)

print("\nClassification Report:")
print(classification_report(
    y_test_true,
    y_pred_hard_vote,
    target_names=CLASS_NAMES,
    digits=4
))

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm_hard_vote,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES
)
plt.title("Hard Vote Ensemble — Top Combination")
plt.ylabel("True")
plt.xlabel("Predicted")
plt.tight_layout()
save_path = os.path.join(RESULT_DIR, "hard_vote_top_combo_cm_internal.png")
plt.savefig(save_path, dpi=150)
print(f"\n💾 Confusion matrix saved → {save_path}")
plt.show()

# External Dataset

## Initiate

In [ ]:
# Zero-Shot & Fine-Tune Hyperparameters 
# Only head layers are trainable, so a lower LR is safer to avoid catastrophic forgetting.
FT_LR       = 1e-5        
FT_EPOCHS   = 100        
FT_PATIENCE = 20          # early stopping patience
FT_BATCH    = 16
FINETUNE_VERSION = "v1"
IMG_SIZE, BATCH    = (224, 224), 32
CLASS_NAMES = ['benign', 'malignant', 'normal']
NUM_CLASSES = 3

# Zero-shot & Fine-tune Model Configurations
DL_MODELS       = ["your_swin_checkpoint", "your_resnet_checkpoint"]          # zero-shot DL backbones
FEATURE_SWIN    = "your_swin_checkpoint"                             # extractor for classical ML models
ML_MODELS       = ["your_lr_checkpoint", "your_svm_checkpoint", "your_rf_checkpoint",
                   "your_xgb_checkpoint", "your_knn_checkpoint", "your_nb_checkpoint", "your_dt_checkpoint"]
# Non-Swin DL zero-shot models: (checkpoint_name, preprocessing)
STD_DL_MODELS = [
      ("your_cnn_checkpoint", "rescale"),  
      ("your_resnet_checkpoint",    "resnet"),    
  ]

class_weights_alpha = [0.35, 0.15, 0.5]  # benign, malignant, normal
focal_loss = tf.keras.losses.CategoricalFocalCrossentropy(
    gamma=2.0, 
    alpha=class_weights_alpha
)
print("Native TF Focal Loss initialized")
# Paths
EXT_ROOT     = "your_external_dataset_root"  # root directory for external dataset
EXT_TRAIN    = os.path.join(EXT_ROOT, "train")
EXT_VAL      = os.path.join(EXT_ROOT, "val")
EXT_TEST_DIR = os.path.join(EXT_ROOT, "test") 
CKPT_DIR   = "/home/benth/TA/checkpoints"
 
EXT_FT_RESULT_DIR = "your_external_finetune_results"  # directory to save fine-tuned model results
ZS_RESULT_DIR = "your_zeroshot_results"  # directory to save zero-shot model results
os.makedirs(EXT_FT_RESULT_DIR, exist_ok=True)
os.makedirs(ZS_RESULT_DIR, exist_ok=True)

print(f"External train : {EXT_TRAIN}")
print(f"External val   : {EXT_VAL}")
print(f"External test  : {EXT_TEST_DIR}")
print(f"Results saved  : {EXT_FT_RESULT_DIR}")
print(f"LR={FT_LR} | EPOCHS={FT_EPOCHS} | PATIENCE={FT_PATIENCE}")

In [ ]:
# Helper
def load_dl_model(path, custom_objects=None):
    return tf.keras.models.load_model(
        path,
        custom_objects=custom_objects or {
            "CategoricalCrossentropy": tf.keras.losses.CategoricalCrossentropy,
        },
    )

def load_swin_model(path):
    model = build_swin_tiny()
    dummy = np.zeros((1, 224, 224, 3), dtype=np.float32)
    _ = model(dummy, training=False)
    with zipfile.ZipFile(path, 'r') as zf:
        with tempfile.TemporaryDirectory() as tmpdir:
            zf.extractall(tmpdir)
            model.load_weights(os.path.join(tmpdir, 'model.weights.h5'))
    return model


def get_dl_preds(model, generator):
    generator.reset()
    return model.predict(generator, verbose=1)

def get_classical_preds(clf_path, features):
    clf = joblib.load(clf_path)
    return clf.predict_proba(features)


def get_malignant_recall(y_true, y_proba):
    y_pred = np.argmax(y_proba, axis=1)
    recalls = recall_score(y_true, y_pred, average=None, zero_division=0)
    return recalls[1] 

def pad_probs(p):
    if p.shape[1] == 2:
        return np.pad(p, ((0, 0), (0, 1)), mode='constant', constant_values=0.0)
    return p

def evaluate_external(name, proba, y_true, save_dir=EXT_FT_RESULT_DIR, verbose=True):
    y_pred = np.argmax(proba, axis=1)
    cm     = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    acc    = np.mean(y_pred == y_true)
    wf1    = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    y_bin  = label_binarize(y_true, classes=[0, 1, 2])

    try:
        auc = roc_auc_score(y_bin, proba, average='macro', multi_class='ovr')
    except ValueError:
        auc = float('nan')

    rec_b = cm[0,0] / cm[0].sum() if cm[0].sum() else 0.0
    rec_m = cm[1,1] / cm[1].sum() if cm[1].sum() else 0.0
    rec_n = cm[2,2] / cm[2].sum() if cm[2].sum() else 0.0

    if verbose:
        print(f"\n{'='*60}")
        print(f"  {name}")
        print(f"{'='*60}")
        print(f"  Accuracy    : {acc:.4f}")
        print(f"  Weighted F1 : {wf1:.4f}")
        print(f"  Macro AUC   : {auc:.4f}")
        print(classification_report(y_true, y_pred, labels=[0, 1, 2],
                                    target_names=CLASS_NAMES, digits=4, zero_division=0))

    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f'{name}\n(External Test)')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    safe = name.replace(' ', '_').replace('/', '-').replace('+', 'plus')
    plt.savefig(os.path.join(save_dir, f"{safe}_cm_ext.png"), dpi=150)
    plt.show()

    return {
        "model":       name,
        "accuracy":    round(acc,  4),
        "weighted_f1": round(wf1,  4),
        "macro_auc":   round(auc,  4),
        "rec_b":       round(rec_b, 4),
        "rec_m":       round(rec_m, 4),
        "rec_n":       round(rec_n, 4),
    }

print("evaluate_external() ready")

def finetune_model(model, model_name, train_gen, val_gen,
                       unfreeze_last_n=10, skip_batchnorm=True, custom_loss=None):
    """
    Fine-tune a loaded Keras model on external data.

    Args:
        model         : loaded Keras model (already trained on internal data)
        model_name    : used for checkpoint filename and logging
        train_gen     : augmented external train generator
        val_gen       : external val generator (no augmentation)
        unfreeze_last_n: number of layers to unfreeze from the top
                        • Use a smaller number than internal training
                        • 0 = head-only fine-tune (safest for tiny datasets)
        skip_batchnorm: keep BatchNorm layers frozen (recommended — prevents
                        running stats from drifting on small external data)

    Returns:
        model         : fine-tuned model (best weights restored)
        history       : Keras History object
    """
    # Freeze all top-level layers 
    for layer in model.layers:
        layer.trainable = False

    # Always unfreeze the classification head (last 5 layers)
    head_layers = [l for l in model.layers if any(
        tag in l.name for tag in ['dense', 'dropout', 'predictions', 'bn_head']
    )]
    for layer in head_layers:
        layer.trainable = True

    # Selectively unfreeze backbone layers
    if unfreeze_last_n > 0:
        # Check if the model has a nested backbone (like Swin or ResNet base)
        backbones = [l for l in model.layers if hasattr(l, 'layers')]

        if backbones:
            base = backbones[0]
            base.trainable = True  # Must be True so gradients flow into it

            # Freeze all inner layers first
            for layer in base.layers:
                layer.trainable = False

            # Unfreeze only the last N inner layers
            for layer in base.layers[-unfreeze_last_n:]:
                is_bn = isinstance(layer, (
                    tf.keras.layers.BatchNormalization,
                    tf.keras.layers.LayerNormalization
                ))
                if skip_batchnorm and is_bn:
                    layer.trainable = False
                else:
                    layer.trainable = True

            trainable_count = sum(1 for l in base.layers if l.trainable) + len(head_layers)
            print(f"  Trainable layers: {trainable_count} (Head + {unfreeze_last_n} inner base layers)")

        else:
            # Flat architecture (like ShallowCNN)
            for layer in model.layers[-unfreeze_last_n:]:
                is_bn = isinstance(layer, (
                    tf.keras.layers.BatchNormalization,
                    tf.keras.layers.LayerNormalization
                ))
                if skip_batchnorm and is_bn:
                    layer.trainable = False
                else:
                    layer.trainable = True

            trainable_count = sum(1 for l in model.layers if l.trainable)
            print(f"  Trainable layers: {trainable_count} / {len(model.layers)}")
    else:
        print(f"  Trainable layers: {len(head_layers)} (Head-only)")

    # Recompile with low LR 
    model.compile(
            optimizer=tf.keras.optimizers.Adam(FT_LR),
            loss=custom_loss if custom_loss is not None else focal_loss,
            metrics=['accuracy']
        )

    # Callbacks 
    ft_ckpt_path = os.path.join(CKPT_DIR, f"{model_name}_ext_ft_{FINETUNE_VERSION}.keras")
    callbacks = [
        EarlyStopping(
            monitor='val_loss', patience=FT_PATIENCE,
            restore_best_weights=True, verbose=1
        ),
        ModelCheckpoint(
            ft_ckpt_path, monitor='val_loss',
            save_best_only=True, verbose=0
        ),
        ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=8, min_lr=1e-8, verbose=1
        ),
    ]

    # Train 
    print(f"  Checkpoint → {ft_ckpt_path}")
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=FT_EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

    print(f" Fine-tune done — best val_loss at epoch "
            f"{np.argmin(history.history['val_loss']) + 1}")

    return model, history

def build_swin_tiny():
    base = SwinTransformerTiny224(include_top=False); base.trainable = False
    i = Input((224, 224, 3), name='pixel_input')
    x = layers.Rescaling(255.0, name='to_uint8')(i)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
    x = layers.Dense(256, activation='relu', name='dense_1')(x)
    x = layers.BatchNormalization(name='bn_head')(x)
    x = layers.Dropout(0.3, name='dropout_1')(x)
    x = layers.Dense(128, activation='relu', name='dense_2')(x)
    x = layers.Dropout(0.2, name='dropout_2')(x)
    o = layers.Dense(NUM_CLASSES, activation='softmax', name='predictions')(x)
    return models.Model(i, o, name='SwinTiny')

def load_swin(name):
    m = build_swin_tiny()
    _ = m(np.zeros((1, 224, 224, 3), np.float32), training=False)
    with zipfile.ZipFile(os.path.join(CKPT_DIR, f"{name}.keras")) as zf:
        with tempfile.TemporaryDirectory() as t:
            zf.extractall(t); m.load_weights(os.path.join(t, 'model.weights.h5'))
    return m

def load_standard(name):
        return tf.keras.models.load_model(os.path.join(CKPT_DIR, f"{name}.keras"), compile=False)

def make_gen(preprocess):
        dg = (ImageDataGenerator(preprocessing_function=resnet_preprocess)
            if preprocess == "resnet" else ImageDataGenerator(rescale=1./255))
        return dg.flow_from_directory(EXT_TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH,
                                    class_mode='categorical', shuffle=False)

def evaluate(name, proba):
    yp = np.argmax(proba, axis=1)
    cm = confusion_matrix(y_true, yp, labels=[0, 1, 2])
    acc = float(np.mean(yp == y_true))
    wf1 = f1_score(y_true, yp, average='weighted', zero_division=0)
    try:
        auc = roc_auc_score(label_binarize(y_true, classes=[0, 1, 2]),
                            proba, average='macro', multi_class='ovr')
    except ValueError:
        auc = float('nan')
    rec = [cm[i, i]/cm[i].sum() if cm[i].sum() else 0.0 for i in range(3)]

    # Print Text Results
    print(f"\n{'='*54}\n  {name}   (zero-shot, external)\n{'='*54}")
    print(f"  acc={acc:.4f} | wF1={wf1:.4f} | AUC={auc:.4f} | "
            f"rec B/M/N = {rec[0]:.3f}/{rec[1]:.3f}/{rec[2]:.3f}")

    # Render Heatmap for Confusion Matrix
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f'{name} — Confusion Matrix')
    plt.ylabel('True Class')
    plt.xlabel('Predicted Class')
    plt.tight_layout()
    plt.savefig(os.path.join(ZS_RESULT_DIR, f"{name}_cm.png"), dpi=150)
    plt.show()

    return {"model": name, "accuracy": round(acc, 4), "weighted_f1": round(wf1, 4),
            "macro_auc": round(acc, 4), "rec_b": round(rec[0], 4),
            "rec_m": round(rec[1], 4), "rec_n": round(rec[2], 4)}

## Zero shot

In [ ]:
# External generator
gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
    EXT_TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH,
    class_mode='categorical', shuffle=False)
y_true = gen.classes
print(f"External test: {gen.samples} imgs | classes={gen.class_indices}")
print(f"Per-class n: " + ", ".join(f"{c}={int((y_true==k).sum())}"
                                   for k, c in enumerate(CLASS_NAMES)))

In [ ]:
rows = []
# DL zero-shot: swin_tiny_v2, swin_tiny_v5
feat_ext, X_ext = None, None
for name in DL_MODELS:
    path = os.path.join(CKPT_DIR, f"{name}.keras")
    if not os.path.exists(path):
        print(f"⚠️  missing {name} — skipping"); continue
    m = load_swin(name)
    gen.reset(); proba = m.predict(gen, verbose=0)
    rows.append(evaluate(name, proba))
    if name == FEATURE_SWIN:                 
        feat_ext = tf.keras.Model(m.input, m.get_layer('global_avg_pool').output)
        gen.reset(); X_ext = feat_ext.predict(gen, verbose=0)
    del m; tf.keras.backend.clear_session()

#Standard DL zero-shot: shallow CNN + ResNet50
for name, prep in STD_DL_MODELS:
      path = os.path.join(CKPT_DIR, f"{name}.keras")
      if not os.path.exists(path):
          print(f"⚠️  missing {name} — skipping"); continue
      m = load_standard(name)
      g = make_gen(prep); g.reset()         
      proba = m.predict(g, verbose=0)
      rows.append(evaluate(name, proba))
      del m; tf.keras.backend.clear_session()
    
# Extract features if FEATURE_SWIN wasn't in DL_MODELS
if X_ext is None:
    m = load_swin(FEATURE_SWIN)
    feat_ext = tf.keras.Model(m.input, m.get_layer('global_avg_pool').output)
    gen.reset(); X_ext = feat_ext.predict(gen, verbose=0)
    del m; tf.keras.backend.clear_session()
print(f"\nExternal Swin features ({FEATURE_SWIN}): {X_ext.shape}")

# ML zero-shot: 7 × *_v6 heads (pipelines self-scale) \
for name in ML_MODELS:
    path = os.path.join(CKPT_DIR, f"{name}.joblib")
    if not os.path.exists(path):
        print(f"⚠️  missing {name} — skipping"); continue
    clf = joblib.load(path)                       
    proba = clf.predict_proba(X_ext)
    rows.append(evaluate(name, proba))

# Leaderboard + save 
df = pd.DataFrame(rows).sort_values("rec_m", ascending=False).reset_index(drop=True)
print("\n" + "="*70)
print("  ZERO-SHOT EXTERNAL — sorted by malignant recall")
print("="*70)
print(df.to_string(index=False))
df.to_csv(os.path.join(ZS_RESULT_DIR, "zeroshot_external_results.csv"), index=False)
print(f"\n Saved → {ZS_RESULT_DIR}/zeroshot_external_results.csv")

# Grouped bar: accuracy + malignant recall 
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
x = np.arange(len(df)); w = 0.6
for ax, (col, title) in zip(axes, [("accuracy", "Accuracy"), ("rec_m", "Malignant Recall")]):
    ax.bar(x, df[col], w, color="#4C72B0")
    ax.set_xticks(x); ax.set_xticklabels(df["model"], rotation=45, ha='right', fontsize=8)
    ax.set_title(f"Zero-shot External — {title}"); ax.set_ylim(0, 1.05)
    ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(ZS_RESULT_DIR, "zeroshot_external.png"), dpi=150)
plt.show()
print("Done.")


## Finetune

In [ ]:
# Preprocessing generator
# Augmentation used for fine-tune train (same as internal training) 
_AUGMENT_KWARGS = dict(
    rotation_range=20, width_shift_range=0.2, height_shift_range=0.2,
    shear_range=0.2, zoom_range=0.2, horizontal_flip=True, fill_mode='nearest'
)
 
def _make_ft_gens(preprocess_fn=None, rescale_val=1./255):
    """
    Returns (train_gen, val_gen, test_gen).
    Train gen has augmentation; val/test gen does not.
    """
    if preprocess_fn:
        train_dg = ImageDataGenerator(preprocessing_function=preprocess_fn, **_AUGMENT_KWARGS)
        eval_dg  = ImageDataGenerator(preprocessing_function=preprocess_fn)
    else:
        train_dg = ImageDataGenerator(rescale=rescale_val, **_AUGMENT_KWARGS)
        eval_dg  = ImageDataGenerator(rescale=rescale_val)
 
    train_gen = train_dg.flow_from_directory(
        EXT_TRAIN, target_size=IMG_SIZE, batch_size=FT_BATCH,
        class_mode='categorical', shuffle=True
    )
    val_gen = eval_dg.flow_from_directory(
        EXT_VAL, target_size=IMG_SIZE, batch_size=FT_BATCH,
        class_mode='categorical', shuffle=False
    )
    test_gen = eval_dg.flow_from_directory(
        EXT_TEST_DIR, target_size=IMG_SIZE, batch_size=FT_BATCH,
        class_mode='categorical', shuffle=False
    )
    return train_gen, val_gen, test_gen

ft_plain_tr,    ft_plain_vl,    ft_plain_ts    = _make_ft_gens(rescale_val=1./255)
ft_resnet_tr,   ft_resnet_vl,   ft_resnet_ts   = _make_ft_gens(resnet_preprocess)
 
y_ext_true = ft_plain_ts.classes   # ground-truth for evaluation
print(f"\n External train  : {ft_plain_tr.samples} samples")
print(f"External val    : {ft_plain_vl.samples} samples")
print(f"External test   : {ft_plain_ts.samples} samples")

### Retrain CNN Model

In [ ]:
# Registry: model_name → (checkpoint, train_gen, val_gen, test_gen, unfreeze_n)
STANDARD_FT_REGISTRY = {
    "resnet50_focal_v4": (
        "resnet50_focal_v4.keras",
        ft_resnet_tr, ft_resnet_vl, ft_resnet_ts,
        0  # Head-only fine-tune
    ),
    "shallow_cnn_focal_v1": (
        "shallow_CNN_focal_v1.keras",
        ft_plain_tr, ft_plain_vl, ft_plain_ts,
        -1   # Unfreeze all layers
    ),
}
 

for model_name, (ckpt_file, tr_gen, vl_gen, ts_gen, unfreeze_n) in STANDARD_FT_REGISTRY.items():
    ckpt_path = os.path.join(CKPT_DIR, ckpt_file)
 
    if not os.path.exists(ckpt_path):
        print(f"⚠️  Skipping {model_name} — checkpoint not found at {ckpt_path}")
        continue
 
    print(f"\n{'='*60}")
    print(f"  Fine-tuning: {model_name}")
    print(f"{'='*60}")
 
    model = load_dl_model(ckpt_path)  # loads with focal loss custom object
 
    # Handle full unfreeze for ShallowCNN
    actual_unfreeze = len(model.layers) if unfreeze_n == -1 else unfreeze_n
 
    model, history = finetune_model(
        model, model_name,
        train_gen=tr_gen, val_gen=vl_gen,
        unfreeze_last_n=actual_unfreeze
    )
 
    # Evaluate on external test
    ts_gen.reset()
    proba  = model.predict(ts_gen, verbose=0)
    result = evaluate_external(f"{model_name}_ext_ft", proba, y_ext_true,
                               save_dir=EXT_FT_RESULT_DIR)

    del model
    tf.keras.backend.clear_session()
 
print("\n All standard models fine-tuned")

### Retrain Swin Model

In [ ]:
FT_BATCH    = 16
SWIN_MODELS_TO_FT = [
    "swin_tiny_v5",        
    "swin_tiny_v2",
]
 
for swin_name in SWIN_MODELS_TO_FT:
    ckpt_path = os.path.join(CKPT_DIR, f"{swin_name}.keras")
 
    if not os.path.exists(ckpt_path):
        print(f"⚠️  Skipping {swin_name} — checkpoint not found")
        continue
 
    print(f"\n{'='*60}")
    print(f"  Fine-tuning: {swin_name}")
    print(f"{'='*60}")
 
    # Rebuild arch + load weights (same pattern as ensemble loader)
    model = build_swin_tiny()
    _ = model(np.zeros((1, 224, 224, 3), dtype=np.float32), training=False)
    with zipfile.ZipFile(ckpt_path, 'r') as zf:
        with tempfile.TemporaryDirectory() as tmp:
            zf.extractall(tmp)
            model.load_weights(os.path.join(tmp, 'model.weights.h5'))
    print("  ✅ Swin weights loaded")
    model, history = finetune_model(
            model, swin_name,
            train_gen=ft_plain_tr, val_gen=ft_plain_vl,
            unfreeze_last_n=10,
            custom_loss=focal_loss  
        )
 
    # Swin fine-tuned checkpoint is saved as .keras by ModelCheckpoint
    # but we also manually save weights in case the zip format is needed
    ft_weights_path = os.path.join(CKPT_DIR, f"{swin_name}_ext_ft_{FINETUNE_VERSION}.weights.h5")
    model.save_weights(ft_weights_path)
    print(f"  ✅ Weights also saved → {ft_weights_path}")
 
    ft_plain_ts.reset()
    proba  = model.predict(ft_plain_ts, verbose=0)
    result = evaluate_external(f"{swin_name}_ext_ft", proba, y_ext_true,
                               save_dir=EXT_FT_RESULT_DIR)
 
    del model
    tf.keras.backend.clear_session()
 
print("\n✅ Swin fine-tuning done")

### Retrain ML Model

In [ ]:
# Load the fine-tuned Swin model using YOUR zipfile workaround
swin_ft_path = os.path.join(CKPT_DIR, f"your_swin_finetuned_checkpoint")

print("Loading Fine-Tuned Swin Model...")
swin_ft_model = build_swin_tiny()
dummy = np.zeros((1, 224, 224, 3), dtype=np.float32)
_ = swin_ft_model(dummy, training=False)   

with zipfile.ZipFile(swin_ft_path, 'r') as zf:
    with tempfile.TemporaryDirectory() as tmpdir:
        zf.extractall(tmpdir)
        swin_ft_model.load_weights(os.path.join(tmpdir, 'model.weights.h5'))

# Build feature extractor
feature_extractor = tf.keras.Model(
    inputs=swin_ft_model.input,
    outputs=swin_ft_model.get_layer('global_avg_pool').output
)
print(f"✅ Feature extractor ready — output dim: {feature_extractor.output_shape}")

# Create strictly non-shuffled generators
eval_dg = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)
gen_train = eval_dg.flow_from_directory(
    EXT_TRAIN, target_size=IMG_SIZE, batch_size=FT_BATCH, class_mode='categorical', shuffle=False
)
gen_val = eval_dg.flow_from_directory(
    EXT_VAL, target_size=IMG_SIZE, batch_size=FT_BATCH, class_mode='categorical', shuffle=False
)
gen_test = eval_dg.flow_from_directory(
    EXT_TEST_DIR, target_size=IMG_SIZE, batch_size=FT_BATCH, class_mode='categorical', shuffle=False
)

# Extract features
print("\nExtracting Features...")
X_train_ft = feature_extractor.predict(gen_train, verbose=1)
y_train_ft = gen_train.classes

X_val_ft = feature_extractor.predict(gen_val, verbose=1)
y_val_ft = gen_val.classes

X_test_ft = feature_extractor.predict(gen_test, verbose=1)
y_test_ft = gen_test.classes

In [ ]:
# Train your ML Heads
print("\nTraining ML Heads")

classifiers_ft = {
    "SVM": SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=42),
    "Random_Forest": RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1),
    "Logistic_Regression": LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, learning_rate=0.05, random_state=42, eval_metric='mlogloss',
verbosity=0),
    "Logistic_Regression": LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=7, metric='cosine'),
}

for name, clf in classifiers_ft.items():
    print(f"\n{'='*40}")
    print(f" {name} (Fine-Tuned Features)")
    print(f"{'='*40}")

    clf.fit(X_train_ft, y_train_ft)
    save_path = os.path.join(CKPT_DIR, f"{name}_ext_ft_{FINETUNE_VERSION}.joblib")
    joblib.dump(clf, save_path)
    print(f"Saved to {save_path}")
    y_pred = clf.predict(X_test_ft)
    print(classification_report(y_test_ft, y_pred, target_names=CLASS_NAMES, labels=[0, 1, 2], digits=4, zero_division=0))
    plt.figure(figsize=(6, 5))
    cm = confusion_matrix(y_test_ft, y_pred, labels=[0, 1, 2])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f'{name} (Fine-Tuned Features) - Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(os.path.join(EXT_FT_RESULT_DIR, f"{name}_cm_ext_ft.png"), dpi=150)
    plt.show()  

## Load Model

In [ ]:
model_name = "your_finetuned_checkpoint"
ver = "v1"

In [ ]:
# Load Model 
model = load_model(
    os.path.join(CKPT_DIR, f"{model_name}_ext_ft_v4.keras"),
    custom_objects={"CategoricalFocalCrossentropy": tf.keras.losses.CategoricalFocalCrossentropy},
    
)
print(f"✅ Model loaded from → {os.path.join(CKPT_DIR, f'{model_name}_ext_ft_v4.keras')}")
model.summary()

In [ ]:
model = build_swin_tiny()   # rebuild the architecture fresh

model.load_weights(os.path.join(CKPT_DIR, f"{model_name}_ext_ft_{ver}.keras"))  # load weights only, skip config deserialization
print(f"✅ Swin-Tiny weights loaded from → {os.path.join(CKPT_DIR, f'{model_name}_ext_ft_{ver}.keras')}")

In [ ]:
proba  = model.predict(ft_plain_ts, verbose=0)
result = evaluate_external(f"{model_name}_ext_ft", proba, y_ext_true,
                            save_dir=EXT_FT_RESULT_DIR)

## Ensemble

In [ ]:
# Helper
def get_swin_probs(model_name): 
    if model_name == 'swin_tiny_v5' :
        ckpt = os.path.join(CKPT_DIR, f"{model_name}_ext_ft_v5.keras") 
    else :
        ckpt = os.path.join(CKPT_DIR, f"{model_name}_ext_ft_v7.keras") 
    swin = build_swin_tiny()
    _ = swin(np.zeros((1, 224, 224, 3), dtype=np.float32), training=False)
    with zipfile.ZipFile(ckpt, 'r') as zf:
        with tempfile.TemporaryDirectory() as tmpdir:
            zf.extractall(tmpdir)
            swin.load_weights(os.path.join(tmpdir, 'model.weights.h5'))

    ft_plain_vl.reset(); val_p = swin.predict(ft_plain_vl, verbose=0)
    ft_plain_ts.reset(); test_p = swin.predict(ft_plain_ts, verbose=0)
    return val_p, test_p

In [ ]:
print(" Extracting DL Probabilities for Ensemble")

print("Loading Swin Tiny v1...")
val_swin_v1, test_swin_v1 = get_swin_probs("your_swin_finetuned_checkpoint")

print("Loading Swin Tiny v2...")
val_swin_v2, test_swin_v2 = get_swin_probs("your_swin_finetuned_checkpoint")

# Load ResNet
print("Loading ResNet50 Focal...")
resnet = load_dl_model(os.path.join(CKPT_DIR, "your_resnet_finetuned_checkpoint"), custom_objects={"CategoricalFocalCrossentropy": tf.keras.losses.CategoricalFocalCrossentropy})
ft_resnet_vl.reset(); val_rn = resnet.predict(ft_resnet_vl, verbose=0)
ft_resnet_ts.reset(); test_rn = resnet.predict(ft_resnet_ts, verbose=0)

In [ ]:
print("Building Meta-Features")

classifiers_ft = {
    "Logistic_Regression": joblib.load(os.path.join(CKPT_DIR, f"your_lr_finetuned_checkpoint.joblib")),
    "Random_Forest":       joblib.load(os.path.join(CKPT_DIR, f"your_rf_finetuned_checkpoint.joblib")),
    "XGBoost":             joblib.load(os.path.join(CKPT_DIR, f"your_xgboost_finetuned_checkpoint.joblib")),
    "SVM":                 joblib.load(os.path.join(CKPT_DIR, f"your_svm_finetuned_checkpoint.joblib")),
    "KNN":                 joblib.load(os.path.join(CKPT_DIR, f"your_knn_finetuned_checkpoint.joblib")),
}

print("✅ Loaded fitted external ML heads")
# Gather all 6 ML Head Probabilities
val_ml_probs, test_ml_probs = [], []
ml_names = ["Logistic_Regression", 
            "Random_Forest", 
            "XGBoost", 
            "SVM", 
            "KNN"]
for name in ml_names:
    clf = classifiers_ft[name]
    val_p = clf.predict_proba(X_val_ft)
    test_p = clf.predict_proba(X_test_ft)

    if val_p.shape[1] == 2:
        val_p = np.pad(val_p, ((0, 0), (0, 1)), mode='constant', constant_values=0.0)
    if test_p.shape[1] == 2:
        test_p = np.pad(test_p, ((0, 0), (0, 1)), mode='constant', constant_values=0.0)

    val_ml_probs.append(val_p)
    test_ml_probs.append(test_p)

# 2. Stack exactly the 10 models you used in Cell 97
X_meta_val = np.concatenate([val_swin_v1, 
                             val_swin_v2, 
                             val_rn] + val_ml_probs, axis=1)
X_meta_test = np.concatenate([test_swin_v1, 
                              test_swin_v2, 
                              test_rn] + test_ml_probs, axis=1)

y_val_true = ft_plain_vl.classes
y_test_true = ft_plain_ts.classes

In [ ]:
# HARD VOTE ENSEMBLE — EXTERNAL TEST DATA
print("HARD VOTE ENSEMBLE — EXTERNAL TEST DATA")


hardvote_probs = {
    # use the combination from internal validation that gave the best malignant recall
    "resnet50_focal": test_rn,
    "XGBoost":          test_ml_probs[2],
    "SVM":              test_ml_probs[3],
    "KNN":              test_ml_probs[4],
}

# predict class labels by majority vote
hard_preds = np.stack(
    [np.argmax(proba, axis=1) for proba in hardvote_probs.values()],
    axis=1
)
y_pred_hardvote = np.array([
    np.bincount(row, minlength=len(CLASS_NAMES)).argmax()
    for row in hard_preds
])

# evaluation
cm_hardvote = confusion_matrix(
    y_test_true,
    y_pred_hardvote,
    labels=list(range(len(CLASS_NAMES)))
)

acc = accuracy_score(y_test_true, y_pred_hardvote)
wf1 = f1_score(y_test_true, y_pred_hardvote, average="weighted", zero_division=0)

print("Models:")
for name in hardvote_probs:
    print(f"  - {name}")

print("\nMetrics:")
print(f"  Accuracy    : {acc:.4f}")
print(f"  Weighted F1 : {wf1:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_test_true,
    y_pred_hardvote,
    target_names=CLASS_NAMES,
    labels=[0, 1, 2],
    digits=4,
    zero_division=0
))

# Plot confusion matrix 
print("Confusion Matrix:")
print(cm_hardvote)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm_hardvote,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES
)
plt.title("Hard Vote Ensemble — External Test Data")
plt.ylabel("True")
plt.xlabel("Predicted")
plt.tight_layout()

os.makedirs(EXT_FT_RESULT_DIR, exist_ok=True)
plt.savefig(os.path.join(EXT_FT_RESULT_DIR, "hard_vote_top_external_cm.png"), dpi=150)
plt.show()

# Save result 
hardvote_row = {
    "ensemble": "hard_vote_top_external",
    "method": "hard_vote",
    "models": "+".join(hardvote_probs.keys()),
    "accuracy": round(acc, 4),
    "weighted_f1": round(wf1, 4),
}

for i, cls in enumerate(CLASS_NAMES):
    hardvote_row[f"{cls}_recall"] = round(
        cm_hardvote[i, i] / cm_hardvote[i].sum(), 4
    )

pd.DataFrame([hardvote_row]).to_csv(
    os.path.join(EXT_FT_RESULT_DIR, "hard_vote_top_external_result.csv"),
    index=False
)

print(f"\nSaved → {EXT_FT_RESULT_DIR}/hard_vote_top_external_result.csv")
print(f"Saved → {EXT_FT_RESULT_DIR}/hard_vote_top_external_cm.png")

In [ ]:
# SOFT VOTING ENSEMBLE — EXTERNAL TEST DATA
print("SOFT VOTING ENSEMBLE — EXTERNAL TEST DATA")

softvote_probs = {
    # use the combination from internal validation that gave the best malignant recall
    "resnet50_focal_v4": test_rn,
    "swin_tiny_v2": test_swin_v2,
    "swin_tiny_v5": test_swin_v1,
    "KNN":              test_ml_probs[4],

}

softvote_weights = {
    # use average weights
    "resnet50_focal_v4": 1/4,
    "swin_tiny_v2": 1/4,
    "swin_tiny_v5": 1/4,
    "KNN": 1/4,
}

# Keep order aligned
model_names = list(softvote_probs.keys())

probs = np.stack(
    [softvote_probs[name] for name in model_names],
    axis=0
)

weights = np.array(
    [softvote_weights[name] for name in model_names],
    dtype=np.float64
)

weights = weights / weights.sum()

print("Models and normalized weights:")
for name, w in zip(model_names, weights):
    print(f"  {name:20s}: {w:.4f}")

# soft voting 
softvote_proba = np.einsum("m,mnc->nc", weights, probs)
y_pred_softvote = np.argmax(softvote_proba, axis=1)

# Evaluation
cm_softvote = confusion_matrix(
    y_test_true,
    y_pred_softvote,
    labels=list(range(len(CLASS_NAMES)))
)

acc = accuracy_score(y_test_true, y_pred_softvote)
wf1 = f1_score(y_test_true, y_pred_softvote, average="weighted", zero_division=0)

print("\nMetrics:")
print(f"  Accuracy    : {acc:.4f}")
print(f"  Weighted F1 : {wf1:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_test_true,
    y_pred_softvote,
    labels=[0, 1, 2],
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
))

print("Confusion Matrix:")
print(cm_softvote)

# Plot confusion matrix 
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm_softvote,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES
)
plt.title("Weighted Soft Voting Ensemble — External Test Data")
plt.ylabel("True")
plt.xlabel("Predicted")
plt.tight_layout()

os.makedirs(EXT_FT_RESULT_DIR, exist_ok=True)
plt.savefig(os.path.join(EXT_FT_RESULT_DIR, "soft_vote_weighted_external_cm.png"), dpi=150)
plt.show()

# Save result 
softvote_row = {
    "ensemble": "soft_vote_weighted_external",
    "method": "weighted_soft_vote",
    "models": "+".join(model_names),
    "weights": "+".join([f"{name}:{softvote_weights[name]:.4f}" for name in model_names]),
    "accuracy": round(acc, 4),
    "weighted_f1": round(wf1, 4),
}

for i, cls in enumerate(CLASS_NAMES):
    softvote_row[f"{cls}_recall"] = round(
        cm_softvote[i, i] / cm_softvote[i].sum(), 4
    )

pd.DataFrame([softvote_row]).to_csv(
    os.path.join(EXT_FT_RESULT_DIR, "soft_vote_weighted_external_result.csv"),
    index=False
)

print(f"\nSaved → {EXT_FT_RESULT_DIR}/soft_vote_weighted_external_result.csv")
print(f"Saved → {EXT_FT_RESULT_DIR}/soft_vote_weighted_external_cm.png")

## Gradcam ++

In [ ]:
# Choose which model to explain: 'resnet50' or 'swin_tiny'
MODEL_TO_ANALYZE = 'swin_tiny'  # Options: 'resnet50', 'swin_tiny'
v = "v1"
print(f"Preparing GradCAM++ analysis for: {MODEL_TO_ANALYZE}...")

if MODEL_TO_ANALYZE == 'resnet50':
    # Load the fine-tuned ResNet50 model
    model_path = os.path.join(CKPT_DIR, "resnet50_focal_v4_ext_ft_v7.keras")
    print(f"Loading ResNet50 model from {model_path}...")
    model = tf.keras.models.load_model(
        model_path,
        custom_objects={
            "CategoricalFocalCrossentropy": tf.keras.losses.CategoricalFocalCrossentropy,
        }
    )
    generator = ft_resnet_ts
    last_conv_layer = None # Auto-detected by GradCAM (conv5_block3_out)

elif MODEL_TO_ANALYZE == 'swin_tiny':
    # Rebuild Swin-Tiny and load fine-tuned weights
    model_name = f"swin_tiny_{v}"
    ckpt_path = os.path.join(CKPT_DIR, f"{model_name}_ext_ft_v5.keras")
    print(f"Rebuilding Swin Tiny and loading weights from {ckpt_path}...")

    model = build_swin_tiny()
    # Run a dummy pass to build shapes before loading weights
    _ = model(np.zeros((1, 224, 224, 3), dtype=np.float32), training=False)

    # Extract weights from the Keras zip file and load
    with zipfile.ZipFile(ckpt_path, 'r') as zf:
        with tempfile.TemporaryDirectory() as tmpdir:
            zf.extractall(tmpdir)
            model.load_weights(os.path.join(tmpdir, 'model.weights.h5'))

    generator = ft_plain_ts
    last_conv_layer = None

# Capture original get_gradcam_pp and temporarily patch it for Swin
_true_gradcam = utils.get_gradcam_pp

def smart_gradcam(model, img, cls, layer=None):
    is_swin = any(
        'swin' in l.name.lower() or 'tiny' in l.name.lower()
        for l in model.layers
    )
    if is_swin:
        return utils.get_gradcam_pp_swin(model, img, cls)
    return _true_gradcam(model, img, cls, layer)

utils.get_gradcam_pp = smart_gradcam

In [ ]:
# Collect Misclassified Samples from the External Test Set 
print("Collecting misclassified samples from external test set...")
misclassified_ext = collect_misclassified(
    model,
    generator,
    class_names=CLASS_NAMES,
    max_per_class=4
)

total_wrong = len(misclassified_ext)
total_test  = generator.samples
print(f"❌ Misclassified on External Test Set: {total_wrong} / {total_test} ({total_wrong/total_test:.1%} error rate)")

if total_wrong > 0:
    # Plot and Save GradCAM++ Grid
    
    if MODEL_TO_ANALYZE == 'swin_tiny':
        output_filename = f"{MODEL_TO_ANALYZE}_ext_ft_gradcam_misclassified_{v}.png"
    else :
        output_filename = f"{MODEL_TO_ANALYZE}_ext_ft_gradcam_misclassified.png"
    save_path = os.path.join(EXT_FT_RESULT_DIR, output_filename)
    print("Plotting GradCAM++ grid for misclassified samples...")
    plot_misclassified_gradcam(
        model,
        misclassified_ext,
        last_conv_layer_name=last_conv_layer,
        max_display=12,
        save_path=save_path,
    )
    print(f"✅ GradCAM++ visualization saved to -> {save_path}")
else:
    print("No misclassified samples to display.")

# Restore the original get_gradcam_pp
utils.get_gradcam_pp = _true_gradcam

# Check Cluster

In [ ]:
# CLUSTER SEPARABILITY ANALYSIS — External Test Set (Benign vs Malignant)
# Prerequisites (must already be in scope from Cell 96 & 98):
#   • X_test_ft, y_test_ft    — GAP embeddings + labels from External fine-tuning section
#   • CLASS_NAMES             — ['benign', 'malignant', 'normal']
#   • EXT_FT_RESULT_DIR       — save path for external results
#   • EXT_TEST_DIR            — path to external test set
#   • _preprocess             — preprocessing dict (e.g. dict(rescale=1./255))

try:
    from sklearn.manifold import TSNE
    HAS_TSNE = True
except ImportError:
    HAS_TSNE = False

CLUSTER_SAVE_DIR = EXT_FT_RESULT_DIR   # Use the same directory for saving cluster analysis results
os.makedirs(CLUSTER_SAVE_DIR, exist_ok=True)

# Colour palette — consistent across all plots
_PALETTE = {
    CLASS_NAMES[0]: "#4C72B0",   # benign  → blue
    CLASS_NAMES[1]: "#DD8452",   # malignant → orange
    CLASS_NAMES[2]: "#55A868",   # normal  → green
}
_MARKERS = {CLASS_NAMES[0]: "o", CLASS_NAMES[1]: "^", CLASS_NAMES[2]: "s"}

print("=" * 60)
print("  CLUSTER SEPARABILITY ANALYSIS (EXTERNAL)")
print("=" * 60)
print(f"  Embedding dim : {X_test_ft.shape[1]}")
print(f"  Test samples  : {X_test_ft.shape[0]}")
print(f"  Classes       : {CLASS_NAMES}")
print("=" * 60)
# Scale embeddings
scaler_viz = StandardScaler()
X_scaled   = scaler_viz.fit_transform(X_test_ft)   # Use data that you want to visualize

# PCA → 50 dims (used for all quantitative metrics)
pca_50 = PCA(n_components=50, random_state=42)
X_pca50 = pca_50.fit_transform(X_scaled)
print(f"\n✅ PCA (50 dims) — variance explained: {pca_50.explained_variance_ratio_.sum():.1%}")

# PCA → 2 dims for visualisation 
pca_2 = PCA(n_components=2, random_state=42)
X_pca2 = pca_2.fit_transform(X_scaled)

# t-SNE → 2 dims for visualisation 
print("Running t-SNE (this may take ~1–2 min) ...")
tsne  = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=1000, init="pca", learning_rate="auto")
X_tsne = tsne.fit_transform(X_pca50)
print("✅ t-SNE done")

# Helper for scatter plots
def _scatter_2d(ax, X2, y, title):
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        mask = y == cls_idx
        ax.scatter(
            X2[mask, 0], X2[mask, 1],
            c=_PALETTE[cls_name], marker=_MARKERS[cls_name],
            alpha=0.55, s=22, linewidths=0, label=cls_name
        )
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel("Component 1"); ax.set_ylabel("Component 2")
    ax.legend(fontsize=8, markerscale=1.5)
    ax.grid(True, linestyle='--', alpha=0.3)

# Combined visualisation panel 
n_plots = 2 
fig, axes = plt.subplots(1, n_plots, figsize=(7 * n_plots, 6))

_scatter_2d(axes[0], X_pca2,  y_test_ft, "PCA (2D)")   # Use data that you want to visualize
_scatter_2d(axes[1], X_tsne,  y_test_ft, "t-SNE (on PCA-50)")   # Use data that you want to visualize


plt.suptitle("Embedding Cluster Separability — External Test Set",
                fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
save_path = os.path.join(CLUSTER_SAVE_DIR, "cluster_2d_projections_external.png")
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"💾 Saved → {save_path}")

In [ ]:
# QUANTITATIVE METRICS (all computed on X_pca50 — NOT on 2D plots)
print("  QUANTITATIVE SEPARABILITY METRICS  (on PCA-50 embeddings)") 

# Silhouette Score (all 3 classes)
sil_all = silhouette_score(X_pca50, y_test_ft, metric='cosine', sample_size=2000, random_state=42)   # Use data that you want to visualize
print(f"\n  Silhouette Score (3-class, cosine)  : {sil_all:.4f}")
print(f"  Interpretation: -1=overlapping  0=touching  +1=separated")

# Silhouette benign vs malignant only
bm_mask = np.isin(y_test_ft, [   # Use data that you want to visualize
    CLASS_NAMES.index('benign'),
    CLASS_NAMES.index('malignant')
])
sil_bm = silhouette_score(X_pca50[bm_mask], y_test_ft[bm_mask], metric='cosine', random_state=42)   # Use data that you want to visualize
print(f"  Silhouette Score (benign vs malignant only) : {sil_bm:.4f}")

# k-Means clustering on PCA-50 → NMI + ARI 
km = KMeans(n_clusters=len(CLASS_NAMES), random_state=42, n_init=10)
km_labels = km.fit_predict(X_pca50)

nmi = normalized_mutual_info_score(y_test_ft, km_labels)   # Use data that you want to visualize
ari = adjusted_rand_score(y_test_ft, km_labels)   # Use data that you want to visualize
print(f"\n  k-Means (k={len(CLASS_NAMES)}) on PCA-50:")
print(f"    NMI (Normalized Mutual Info) : {nmi:.4f}  (0=random, 1=perfect)")
print(f"    ARI (Adjusted Rand Index)    : {ari:.4f}  (0=random, 1=perfect)")

# Folder-Discriminator AUC (benign vs malignant) 
print("\n  Folder-Discriminator AUC (benign vs malignant):")
bm_X   = X_pca50[bm_mask]
bm_y   = (y_test_ft[bm_mask] == CLASS_NAMES.index('malignant')).astype(int)   # Use data that you want to visualize

from sklearn.model_selection import StratifiedShuffleSplit
sss = StratifiedShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
disc_aucs = []
for train_idx, val_idx in sss.split(bm_X, bm_y):
    lr_disc = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
    lr_disc.fit(bm_X[train_idx], bm_y[train_idx])
    prob = lr_disc.predict_proba(bm_X[val_idx])[:, 1]
    disc_aucs.append(roc_auc_score(bm_y[val_idx], prob))

disc_auc_mean = np.mean(disc_aucs)
disc_auc_std  = np.std(disc_aucs)
print(f"  Discriminator AUC (5-fold CV): {disc_auc_mean:.4f} ± {disc_auc_std:.4f}")

if disc_auc_mean >= 0.85:
    verdict = "✅ HIGH — benign and malignant are representationally distinct"
elif disc_auc_mean >= 0.65:
    verdict = "⚠️  MODERATE — partial overlap; some samples look similar"
else:
    verdict = "🔴 LOW — embeddings are nearly indistinguishable"
print(f"  Verdict: {verdict}")